<a href="https://colab.research.google.com/github/Phantom-dcode/reasoning-preserving-quantization/blob/main/quant_reasoning_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Quantization × reasoning: full experiment run
Runs everything the reviewers asked for, on a Colab **T4 GPU**:
1. A validated harness (chat template, `\boxed{}` extraction, 512 new tokens) on the **full GSM8K test set (1,319)**.
2. fp16, uniform NF4 and standalone INT8 baselines.
3. C1 mixed precision with an **audit** that proves the protected blocks really stay in fp16 (memory + module types).
4. C2 verification-gated fallback with a multi-line arithmetic checker (computed from the NF4 and INT8 runs).
5. C3 clamping (CoT and generic calibration), the combination, and the **H1/H2 ablations**
   (single-layer, first-k, massive-dimension exemption, constrained max-clamp with joint calibration, KL divergence).
6. Wilson intervals, exact McNemar tests and power, written to `results.json`.

**How to run:** Runtime → Change runtime type → T4 GPU. Run the install cell, then *Runtime → Run all*.
First run with `SMOKE = True` (≈10 min) to check that fp16 accuracy is near 70%; then set `SMOKE = False`.
Results are saved after every configuration, so a disconnected session resumes where it stopped.

In [ ]:
# Install pinned libraries (about 1 minute)
!pip -q install "transformers==5.17.0" "bitsandbytes>=0.45" accelerate datasets scipy
import torch; print(torch.__version__, "GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.9 MB/s eta 0:00:00
2.11.0+cu128 GPU: Tesla T4


In [ ]:
import os, re, json, math, gc, time, ast, operator, random, platform, contextlib
import numpy as np
import torch
from scipy import stats as sps
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, GenerationConfig

TEST_MODE = os.environ.get("QR_TEST_MODE") == "1"   # offline pipeline test only
SMOKE = True                                          # True = 16 items per config, for a quick check

CFG = dict(
    model_id="Qwen/Qwen2.5-1.5B-Instruct",
    n_main=None,            # None = all 1,319 GSM8K test items
    n_ablation=330,         # items per ablation run (80% power for a 10-point difference)
    batch_size=64,
    max_new_tokens=512,
    system_prompt="Please reason step by step, and put your final answer within \\boxed{}.",
    seed=42,
    n_calib=30,             # calibration traces (GSM8K train) for C1 probe and C3 thresholds
    n_heldout=30,           # held-out traces for KL divergence
    noise_scale=0.1,        # C1 probe: noise std = noise_scale x std of the block output
    top_frac=0.25,          # C1: fraction of blocks kept in fp16
    clamp_q=0.995,          # C3 percentile
    single_layers=[0, 1, 2, 4, 14, 27],
    first_k=[2, 4, 8, 16],
    exempt_top_m=8,         # A3: dimensions with the largest calibration max|x| exempted per layer
    constrained_k=[3, 6],   # A4: C = max(P99.5, mu + k*sigma), joint calibration
    reference_fp16_acc=73.2,
    save_dir="/content/drive/MyDrive/quant_reasoning_runs",
)
if SMOKE:
    CFG.update(n_main=16, n_ablation=8, save_dir=CFG["save_dir"] + "_smoke")
if TEST_MODE:
    CFG.update(model_id=os.environ["QR_TINY_MODEL"], n_main=10, n_ablation=6, batch_size=4, max_new_tokens=12,
               n_calib=3, n_heldout=3, single_layers=[0, 3], first_k=[2], exempt_top_m=2,
               save_dir=os.environ["QR_SAVE_DIR"])

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])

if not TEST_MODE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive not mounted, saving locally:", e)
        CFG["save_dir"] = CFG["save_dir"].replace("/content/drive/MyDrive", "/content")
SAVE = CFG["save_dir"]
os.makedirs(SAVE, exist_ok=True)
RESULTS_PATH = os.path.join(SAVE, "results.json")
RESULTS = json.load(open(RESULTS_PATH)) if os.path.exists(RESULTS_PATH) else {}


def save_results():
    json.dump(RESULTS, open(RESULTS_PATH, "w"), indent=1, default=float)


def env_info():
    import importlib
    info = dict(python=platform.python_version(), torch=torch.__version__, transformers=transformers.__version__,
                device=DEVICE, gpu=torch.cuda.get_device_name(0) if DEVICE == "cuda" else None)
    for m in ["bitsandbytes", "accelerate", "datasets"]:
        try:
            info[m] = importlib.import_module(m).__version__
        except Exception:
            info[m] = None
    return info


RESULTS["env"] = env_info(); RESULTS["config"] = CFG; save_results()
print(json.dumps(RESULTS["env"], indent=1))

Mounted at /content/drive
{
 "python": "3.13.15",
 "torch": "2.11.0+cu128",
 "transformers": "5.17.0",
 "device": "cuda",
 "gpu": "Tesla T4",
 "bitsandbytes": "0.50.2",
 "accelerate": "1.14.0",
 "datasets": "4.8.5"
}


## Data, prompt and answer extraction

In [ ]:
def norm_num(s):
    if s is None:
        return None
    s = str(s).strip().replace(",", "").replace("$", "").replace("%", "").rstrip(".")
    try:
        return float(s)
    except ValueError:
        return None


def load_gsm8k(split):
    try:
        from datasets import load_dataset
        ds = load_dataset("openai/gsm8k", "main", split=split)
        rows, src = [dict(question=r["question"], answer=r["answer"]) for r in ds], "hf:openai/gsm8k"
    except Exception:
        import urllib.request
        url = ("https://raw.githubusercontent.com/openai/grade-school-math/master/"
               f"grade_school_math/data/{split}.jsonl")
        rows = [json.loads(l) for l in urllib.request.urlopen(url).read().decode().splitlines() if l.strip()]
        src = url
    for i, r in enumerate(rows):
        r["idx"] = i
        r["gold"] = norm_num(r["answer"].split("####")[-1])
    return rows, src


def load_generic(n):
    try:
        from datasets import load_dataset
        ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
        paras = [t.strip() for t in ds["text"] if len(t.strip()) > 400 and not t.strip().startswith("=")]
        return paras[:n], "hf:wikitext-2-raw-v1"
    except Exception:
        if not TEST_MODE:
            raise
        return ["The river flows through the old town and past the market square."] * n, "test-fallback"


test_rows, src_test = load_gsm8k("test")
train_rows, src_train = load_gsm8k("train")
rng = np.random.default_rng(CFG["seed"])
perm = rng.permutation(len(test_rows)).tolist()
MAIN = test_rows if CFG["n_main"] is None else [test_rows[i] for i in perm[:CFG["n_main"]]]
ABL = [test_rows[i] for i in perm[:CFG["n_ablation"]]]
tperm = rng.permutation(len(train_rows)).tolist()
CALIB = [train_rows[i] for i in tperm[:CFG["n_calib"]]]
HELDOUT = [train_rows[i] for i in tperm[CFG["n_calib"]:CFG["n_calib"] + CFG["n_heldout"]]]
GENERIC, src_generic = load_generic(CFG["n_calib"])
RESULTS["data"] = dict(test_source=src_test, generic_source=src_generic, n_main=len(MAIN), n_ablation=len(ABL),
                       ablation_idx=[r["idx"] for r in ABL], calib_idx=[r["idx"] for r in CALIB])
save_results()
print(f"main={len(MAIN)} ablation={len(ABL)} calib={len(CALIB)} heldout={len(HELDOUT)}")


def last_boxed(text):
    i = text.rfind("\\boxed{")
    if i < 0:
        return None
    j = k = i + len("\\boxed{")
    depth = 1
    while k < len(text) and depth:
        depth += {"{": 1, "}": -1}.get(text[k], 0)
        k += 1
    return text[j:k - 1] if depth == 0 else None


NUM_RE = re.compile(r"-?\d[\d,]*(?:\.\d+)?")


def extract_pred(text):
    b = last_boxed(text)
    src = b if b is not None else text
    src = re.sub(r"\\frac\{([^{}]*)\}\{([^{}]*)\}", r"\1/\2", src)
    nums = NUM_RE.findall(src)
    if not nums:
        return None, b is not None
    if b is not None and re.fullmatch(r"\s*-?\d+(?:\.\d+)?\s*/\s*\d+(?:\.\d+)?\s*", src):
        a, c = src.split("/")
        return float(a) / float(c), True
    return norm_num(nums[-1]), b is not None


def build_prompt(tok, question):
    msgs = [{"role": "system", "content": CFG["system_prompt"]}, {"role": "user", "content": question}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


def trace_text(tok, row):
    sol = re.sub(r"<<[^>]*>>", "", row["answer"].split("####")[0]).strip()
    ans = row["answer"].split("####")[-1].strip()
    msgs = [{"role": "system", "content": CFG["system_prompt"]}, {"role": "user", "content": row["question"]},
            {"role": "assistant", "content": f"{sol}\nThe final answer is \\boxed{{{ans}}}."}]
    return tok.apply_chat_template(msgs, tokenize=False)

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

...

wikitext-2-raw-v1/validation-00000-of-00(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

main=16 ablation=8 calib=30 heldout=30


In [ ]:
def load_tokenizer():
    tok = AutoTokenizer.from_pretrained(CFG["model_id"])
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok


TOK = load_tokenizer()


def load_model(kind, skip=None):
    kw = dict(dtype=DTYPE, device_map=DEVICE)
    if kind == "nf4":
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DTYPE,
            llm_int8_skip_modules=skip)
    elif kind == "int8":
        kw["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
    model = AutoModelForCausalLM.from_pretrained(CFG["model_id"], **kw)
    model.eval()
    return model


def free(model):
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


def footprint_gb(model):
    return model.get_memory_footprint() / 1e9


def layers_of(model):
    return model.model.layers


def eos_ids(model):
    ids = model.generation_config.eos_token_id
    ids = set(ids if isinstance(ids, (list, tuple)) else [ids])
    ids.add(TOK.eos_token_id)
    ids.add(TOK.pad_token_id)
    return {i for i in ids if i is not None}


@contextlib.contextmanager
def hooks(model, fns):
    handles = [layers_of(model)[l].register_forward_hook(f) for l, f in fns.items()]
    try:
        yield
    finally:
        for h in handles:
            h.remove()


def _out(out, h):
    return (h,) + tuple(out[1:]) if isinstance(out, tuple) else h


def _hidden(out):
    return out[0] if isinstance(out, tuple) else out


def clamp_hook(C, exempt=None):
    def fn(mod, inp, out):
        h = _hidden(out)
        hc = h.clamp(-C, C)
        if exempt is not None:
            hc = torch.where(exempt.to(h.device), h, hc)
        return _out(out, hc)
    return fn


@torch.no_grad()
def generate(model, questions):
    stop = eos_ids(model)
    model.generation_config = GenerationConfig(
        do_sample=False, num_beams=1, max_new_tokens=CFG["max_new_tokens"], repetition_penalty=1.0,
        pad_token_id=TOK.pad_token_id, eos_token_id=sorted(stop))
    prompts = [build_prompt(TOK, q) for q in questions]
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    out = [None] * len(prompts)
    bs = CFG["batch_size"]
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        enc = TOK([prompts[i] for i in idx], return_tensors="pt", padding=True).to(model.device)
        gen = model.generate(**enc)
        new = gen[:, enc["input_ids"].shape[1]:].tolist()
        for row, i in zip(new, idx):
            n = next((p for p, t in enumerate(row) if t in stop), len(row))
            out[i] = (TOK.decode(row[:n], skip_special_tokens=True), n)
    return out


def run_config(name, model, items, clamp_fns=None, meta=None):
    path = os.path.join(SAVE, f"{name}.jsonl")
    if os.path.exists(path):
        recs = [json.loads(l) for l in open(path)]
        if len(recs) == len(items):
            print(f"[{name}] loaded {len(recs)} saved records")
            return recs
    t0 = time.time()
    with hooks(model, clamp_fns or {}):
        gens = generate(model, [r["question"] for r in items])
    recs = []
    for r, (text, n) in zip(items, gens):
        pred, boxed = extract_pred(text)
        recs.append(dict(idx=r["idx"], gold=r["gold"], pred=pred, boxed=boxed, n_tokens=n, text=text,
                         correct=bool(pred is not None and r["gold"] is not None and abs(pred - r["gold"]) < 1e-4)))
    with open(path, "w") as f:
        for rec in recs:
            f.write(json.dumps(rec) + "\n")
    info = dict(seconds=round(time.time() - t0, 1), footprint_gb=round(footprint_gb(model), 3))
    info.update(meta or {})
    RESULTS.setdefault("run_meta", {})[name] = info
    save_results()
    acc = 100 * np.mean([r["correct"] for r in recs])
    print(f"[{name}] acc={acc:.1f}% n={len(recs)} {info}")
    return recs


def wilson(k, n, z=1.959963984540054):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z / d * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (100 * max(0.0, c - h), 100 * min(1.0, c + h))


def mcnemar(ref, other):
    """Exact two-sided McNemar test on items shared by both runs (matched by idx)."""
    a = {r["idx"]: r["correct"] for r in ref}
    b = {r["idx"]: r["correct"] for r in other}
    common = sorted(set(a) & set(b))
    b_ = sum(a[i] and not b[i] for i in common)
    c_ = sum(b[i] and not a[i] for i in common)
    p = 1.0 if b_ + c_ == 0 else float(min(1.0, sps.binomtest(min(b_, c_), b_ + c_, 0.5).pvalue))
    return dict(ref_only=b_, other_only=c_, p=p, n=len(common))


def summarize(recs):
    k, n = sum(r["correct"] for r in recs), len(recs)
    lo, hi = wilson(k, n)
    return dict(n=n, correct=k, acc=100 * k / n, ci=[lo, hi], cot_tokens=float(np.mean([r["n_tokens"] for r in recs])),
                boxed_rate=100 * float(np.mean([r["boxed"] for r in recs])),
                hit_max_tokens=100 * float(np.mean([r["n_tokens"] >= CFG["max_new_tokens"] for r in recs])))


def power_two_prop(p1, p2, n, alpha=0.05):
    z = sps.norm.ppf(1 - alpha / 2)
    pb = (p1 + p2) / 2
    se0 = math.sqrt(2 * pb * (1 - pb) / n)
    se1 = math.sqrt(p1 * (1 - p1) / n + p2 * (1 - p2) / n)
    d = abs(p2 - p1)
    return sps.norm.cdf((d - z * se0) / se1) + sps.norm.cdf((-d - z * se0) / se1)


RUNS = {}

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [ ]:

# Select one question from the main dataset
test_question = MAIN[0]["question"]

print("Test question:", test_question)

# 1. NF4 baseline (no clamping)
baseline = generate(model, [test_question])

print("\n--- NF4 BASELINE ---")
print(baseline[0][0])
print("Generated tokens:", baseline[0][1])


# 2. Clamp only layer 25
clamp_25 = {
    25: clamp_hook(C3_COT[25].__closure__[0].cell_contents)
}

with hooks(model, clamp_25):
    result_25 = generate(model, [test_question])

print("\n--- LAYER 25 CLAMPING ---")
print(result_25[0][0])
print("Generated tokens:", result_25[0][1])


# 3. Clamp only layer 27
clamp_27 = {
    27: clamp_hook(C3_COT[27].__closure__[0].cell_contents)
}

with hooks(model, clamp_27):
    result_27 = generate(model, [test_question])

print("\n--- LAYER 27 CLAMPING ---")
print(result_27[0][0])
print("Generated tokens:", result_27[0][1])

Test question: Darrell and Allen's ages are in the ratio of 7:11. If their total age now is 162, calculate Allen's age 10 years from now.

--- NF4 BASELINE ---
Let's denote Darrell's age as \(7x\) and Allen's age as \(11x\), where \(x\) is a common multiplier. According to the problem, the sum of their ages is 162, so we can write the equation:

\[7x + 11x = 162\]

Combining like terms, we get:

\[18x = 162\]

To find \(x\), we divide both sides by 18:

\[x = \frac{162}{18} = 9\]

Now that we have \(x\), we can find the current ages of both Darrell and Allen:

- Darrell's age is \(7x = 7 \times 9 = 63\)
- Allen's age is \(11x = 11 \times 9 = 99\)

To find Allen's age 10 years from now, we add 10 to his current age:

\[99 + 10 = 109\]

Therefore, Allen's age 10 years from now is \(\boxed{109}\).
Generated tokens: 245

--- LAYER 25 CLAMPING ---
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

In [ ]:

test_layers = [24, 26]

for layer_id in test_layers:
    print(f"\n{'='*50}")
    print(f"Testing layer {layer_id}")
    print(f"{'='*50}")

    threshold = C3_COT[layer_id].__closure__[0].cell_contents

    single_clamp = {
        layer_id: clamp_hook(threshold)
    }

    with hooks(model, single_clamp):
        result = generate(model, [test_question])

    print("Generated output:")
    print(result[0][0])

    print("\nGenerated tokens:", result[0][1])


Testing layer 24
Generated output:
To determine Darbleel and Allen's current ages, we start by setting up the given ratio and the total age equation.

The ratio of Allen's age to Allen's age is 11 (since Darbleel is 7 and Darbleel is 11).

Let \( A_1 \) be Allen's current age.
Let **\( D \)** be Allen's current age.

From the ratio, we have:
\[
\frac{A_1}{11} = \frac{7}{11}
\]

By cross-multiplying, we find:
\[
A_1 = 7
\]

Therefore, Allen's age is **7 years**.

Given the combined age is 162, we can write the total age equation as follows:
\[
7 + D = 162
\]

By solving for \( D \), we get:
\[
7 + D = 162
\]
\[
111.1111 = 111.1111
\]

Therefore, Allen's age is:
\[
\boxed{111.1111}
\]

To find out how old Allen will be in 10 years, we simply add 10 to his current age:
\[
111.1111 + 10 = 121.1111
\]

Therefore, in 10 years, Allen's age will be **121.1111 years**.

Generated tokens: 321

Testing layer 26
Generated output:
To determine Allen's possible future babysitting babysitting babysi

In [ ]:
# Controlled threshold experiment

model = load_model("nf4")

thresholds = [8, 16, 32]

threshold_results = {}

for multiplier in thresholds:

    threshold = st_cot[25]["p995"] * multiplier

    name = f"c3_layer25_{multiplier}x"

    print(f"\nRunning {name}")
    print(f"Threshold: {threshold:.4f}")

    intervention = {
        25: clamp_hook(threshold)
    }

    result = run_config(
        name,
        model,
        MAIN,
        intervention
    )

    threshold_results[name] = result

    print("Result:", result)

free(model)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


Running c3_layer25_8x
Threshold: 129.0000
[c3_layer25_8x] acc=62.5% n=16 {'seconds': 27.7, 'footprint_gb': 1.122}
Result: [{'idx': 30, 'gold': 109.0, 'pred': 109.0, 'boxed': True, 'n_tokens': 260, 'text': "Let's denote the age of Darrell as \\(7x\\) and the age of Allen as \\(11x\\), where \\(x\\) is a common multiplier. According to the problem, the sum of their ages is 162. Therefore, we can write the equation:\n\n\\[7x + 11x = 162\\]\n\nCombining the terms on the left side, we get:\n\n\\[18x = 162\\]\n\nTo find \\(x\\), we divide both sides by 18:\n\n\\[x = \\frac{162}{18} = 9\\]\n\nNow that we know \\(x = 9\\), we can determine the current age of each person. The age of Darrell is:\n\n\\[7x = 7 \\times 9 = 63\\]\n\nAnd the age of Allen is:\n\n\\[11x = 11 \\times 9 = 99\\]\n\nTo find out how old Allen will be 10 years from now, we add 10 to his current age:\n\n\\[99 + 10 = 109\\]\n\nTherefore, the age of Allen 10 years from now will be \\(\\boxed{109}\\).", 'correct': True}, {'idx'

In [ ]:
print("Result type:", type(result))
print("Result:", repr(result)[:2000])

Result type: <class 'list'>
Result: [("To determine Darbleel and Allen's current ages, we start by setting up the given ratio and the total age equation.\n\nThe ratio of Allen's age to Allen's age is 11 (since Darbleel is 7 and Darbleel is 11).\n\nLet \\( A_1 \\) be Allen's current age.\nLet **\\( D \\)** be Allen's current age.\n\nFrom the ratio, we have:\n\\[\n\\frac{A_1}{11} = \\frac{7}{11}\n\\]\n\nBy cross-multiplying, we find:\n\\[\nA_1 = 7\n\\]\n\nTherefore, Allen's age is **7 years**.\n\nGiven the combined age is 162, we can write the total age equation as follows:\n\\[\n7 + D = 162\n\\]\n\nBy solving for \\( D \\), we get:\n\\[\n7 + D = 162\n\\]\n\\[\n111.1111 = 111.1111\n\\]\n\nTherefore, Allen's age is:\n\\[\n\\boxed{111.1111}\n\\]\n\nTo find out how old Allen will be in 10 years, we simply add 10 to his current age:\n\\[\n111.1111 + 10 = 121.1111\n\\]\n\nTherefore, in 10 years, Allen's age will be **121.1111 years**.", 321)]


In [ ]:
test_layers = [24, 26]

for layer_id in test_layers:
    threshold = C3_COT[layer_id].__closure__[0].cell_contents
    single_clamp = {layer_id: clamp_hook(threshold)}

    with hooks(model, single_clamp):
        result = generate(model, [test_question])

    print(f"\nLayer {layer_id}")
    print("Threshold:", threshold)
    print("Output:", result[0][0][:1000])
    print("Generated tokens:", result[0][1])


Layer 24
Threshold: 14.609375
Output: To determine Darbleel and Allen's current ages, we start by setting up the given ratio and the total age equation.

The ratio of Allen's age to Allen's age is 11 (since Darbleel is 7 and Darbleel is 11).

Let \( A_1 \) be Allen's current age.
Let **\( D \)** be Allen's current age.

From the ratio, we have:
\[
\frac{A_1}{11} = \frac{7}{11}
\]

By cross-multiplying, we find:
\[
A_1 = 7
\]

Therefore, Allen's age is **7 years**.

Given the combined age is 162, we can write the total age equation as follows:
\[
7 + D = 162
\]

By solving for \( D \), we get:
\[
7 + D = 162
\]
\[
111.1111 = 111.1111
\]

Therefore, Allen's age is:
\[
\boxed{111.1111}
\]

To find out how old Allen will be in 10 years, we simply add 10 to his current age:
\[
111.1111 + 10 = 121.1111
\]

Therefore, in 10 years, Allen's age will be **121.1111 years**.
Generated tokens: 321

Layer 26
Threshold: 17.421875
Output: To determine Allen's possible future babysitting babysitting b

In [ ]:
# Generate without any clamp
baseline_result = generate(model, [test_question])

print("Baseline output:")
print(baseline_result[0][0][:1000])

print("\nGenerated tokens:", baseline_result[0][1])

Baseline output:
Let's denote Darrell's age as \(7x\) and Allen's age as \(11x\), where \(x\) is a common multiplier. According to the problem, the sum of their ages is 162, so we can write the equation:

\[7x + 11x = 162\]

Combining like terms, we get:

\[18x = 162\]

To find \(x\), we divide both sides by 18:

\[x = \frac{162}{18} = 9\]

Now that we have \(x\), we can find the current ages of both Darrell and Allen:

- Darrell's age is \(7x = 7 \times 9 = 63\)
- Allen's age is \(11x = 11 \times 9 = 99\)

To find Allen's age 10 years from now, we add 10 to his current age:

\[99 + 10 = 109\]

Therefore, Allen's age 10 years from now is \(\boxed{109}\).

Generated tokens: 245


In [ ]:
test_layers = [24, 26]

for layer_id in test_layers:
    threshold = (
        C3_COT[layer_id].__closure__[0].cell_contents
    )

    print(f"Layer {layer_id}")
    print(f"Threshold: {threshold}")

Layer 24
Threshold: 14.609375
Layer 26
Threshold: 17.421875


In [ ]:
import torch

def make_counting_clamp(C, stats):
    def fn(mod, inp, out):
        h = _hidden(out)

        total = h.numel()
        clipped = (h.abs() > C).sum().item()

        stats["total"] += total
        stats["clipped"] += clipped

        hc = h.clamp(-C, C)
        return _out(out, hc)

    return fn


test_layers = [24, 26]

for layer_id in test_layers:
    threshold = C3_COT[layer_id].__closure__[0].cell_contents

    stats = {"total": 0, "clipped": 0}

    counting_hook = {
        layer_id: make_counting_clamp(threshold, stats)
    }

    with hooks(model, counting_hook):
        result = generate(model, [test_question])

    clipping_rate = (
        100 * stats["clipped"] / stats["total"]
        if stats["total"] > 0 else 0
    )

    print(f"\nLayer {layer_id}")
    print(f"Threshold: {threshold}")
    print(f"Total activation values: {stats['total']:,}")
    print(f"Clipped activation values: {stats['clipped']:,}")
    print(f"Clipping rate: {clipping_rate:.6f}%")
    print(f"Generated tokens: {result[0][1]}")
    print(f"Output: {result[0][0][:300]}")


Layer 24
Threshold: 14.609375
Total activation values: 595,968
Clipped activation values: 2,846
Clipping rate: 0.477542%
Generated tokens: 321
Output: To determine Darbleel and Allen's current ages, we start by setting up the given ratio and the total age equation.

The ratio of Allen's age to Allen's age is 11 (since Darbleel is 7 and Darbleel is 11).

Let \( A_1 \) be Allen's current age.
Let **\( D \)** be Allen's current age.

From the ratio, 

Layer 26
Threshold: 17.421875
Total activation values: 125,952
Clipped activation values: 781
Clipping rate: 0.620077%
Generated tokens: 15
Output: To determine Allen's possible future babysitting babysitting babysitting babysitting.


## Generation, hooks and statistics

## Stage 1 — fp16: harness validation, C1 sensitivity probe

In [ ]:
@torch.no_grad()
def seq_loss(model, text):
    ids = TOK(text, return_tensors="pt").input_ids.to(model.device)
    return model(input_ids=ids, labels=ids).loss.float().item()


@torch.no_grad()
def sensitivity_probe(model):
    texts = [trace_text(TOK, r) for r in CALIB]
    base = float(np.mean([seq_loss(model, t) for t in texts]))
    scores = []
    for l in range(len(layers_of(model))):
        gen = torch.Generator(device=model.device).manual_seed(CFG["seed"])

        def fn(mod, inp, out, gen=gen):
            h = _hidden(out)
            noise = torch.randn(h.shape, generator=gen, device=h.device, dtype=torch.float32)
            return _out(out, h + (noise * CFG["noise_scale"] * h.float().std()).to(h.dtype))
        with hooks(model, {l: fn}):
            scores.append(float(np.mean([seq_loss(model, t) for t in texts])) - base)
    return base, scores


model = load_model("fp16")
RUNS["fp16"] = run_config("fp16", model, MAIN)
if "sensitivity" not in RESULTS:
    base, scores = sensitivity_probe(model)
    n_keep = max(1, round(CFG["top_frac"] * len(scores)))
    RESULTS["sensitivity"] = dict(base_loss=base, delta_loss=scores,
                                  protected=sorted(np.argsort(scores)[::-1][:n_keep].tolist()))
    save_results()
fp16_acc = summarize(RUNS["fp16"])["acc"]
RESULTS["harness_check"] = dict(fp16_acc=fp16_acc, reference=CFG["reference_fp16_acc"],
                                ok=bool(fp16_acc >= CFG["reference_fp16_acc"] - 10))
save_results()
print("Protected blocks for C1:", RESULTS["sensitivity"]["protected"])
if not RESULTS["harness_check"]["ok"] and not TEST_MODE:
    print("WARNING: fp16 accuracy is more than 10 points below the published 73.2%. "
          "Inspect a few outputs in fp16.jsonl before trusting the other results.")
free(model)

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[fp16] acc=68.8% n=16 {'seconds': 22.7, 'footprint_gb': 3.087}
Protected blocks for C1: [2, 4, 5, 6, 7, 12, 13]


## Stage 2 — standalone INT8 (the C2 fallback model)

In [ ]:
model = load_model("int8")
RUNS["int8"] = run_config("int8", model, MAIN)
free(model)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[int8] acc=87.5% n=16 {'seconds': 92.9, 'footprint_gb': 1.777}


## Stage 3 — uniform NF4, massive-activation probe, C3 and ablations

In [ ]:
@torch.no_grad()
def layer_stats(model, texts, only=None, clamp_fns=None):
    """Per-layer |x| statistics of decoder-layer outputs."""

    L = len(layers_of(model))
    targets = range(L) if only is None else [only]

    buf = {l: [] for l in targets}
    dmax = {l: None for l in targets}

    def mk(l):
        def fn(mod, inp, out):
            a = _hidden(out).detach().abs().float()[0]

            # Keep calibration statistics in FP32
            buf[l].append(a.flatten().float().cpu())

            m = a.amax(dim=0).cpu()
            dmax[l] = (
                m if dmax[l] is None
                else torch.maximum(dmax[l], m)
            )

        return fn

    fns = dict(clamp_fns or {})

    for l in targets:
        if l in fns:
            prev = fns[l]

            def both(mod, inp, out, prev=prev, rec=mk(l)):
                rec(mod, inp, out)
                return prev(mod, inp, out)

            fns[l] = both
        else:
            fns[l] = mk(l)

    with hooks(model, fns):
        for t in texts:
            model(
                input_ids=TOK(t, return_tensors="pt").input_ids.to(
                    model.device
                )
            )

    stats = {}

    for l in targets:
        v = torch.cat(buf[l]).float()

        k = max(
            1,
            int(math.ceil(CFG["clamp_q"] * v.numel()))
        )

        p = float(torch.kthvalue(v, k).values)

        stats[l] = dict(
            p995=p,
            mean=float(v.mean()),
            std=float(v.std()),
            median=float(v.median()),
            max=float(v.max()),
            top_dims=torch.topk(
                dmax[l], min(8, dmax[l].numel())
            ).indices.tolist(),
            top_vals=torch.topk(
                dmax[l], min(8, dmax[l].numel())
            ).values.tolist(),
            dim_max=dmax[l]
        )

    return stats


@torch.no_grad()
def mean_kl(model, clamp_fns):
    """Mean token KL(P_unclamped || P_clamped) on held-out traces for the same model."""
    tot, n = 0.0, 0
    for r in HELDOUT:
        ids = TOK(trace_text(TOK, r), return_tensors="pt").input_ids.to(model.device)
        lp = torch.log_softmax(model(input_ids=ids).logits.float(), -1)
        with hooks(model, clamp_fns):
            lq = torch.log_softmax(model(input_ids=ids).logits.float(), -1)
        tot += float((lp.exp() * (lp - lq)).sum(-1).sum())
        n += ids.shape[1]
    return tot / n


model = load_model("nf4")
RUNS["nf4"] = run_config("nf4", model, MAIN)
L = len(layers_of(model))
cot_texts = [trace_text(TOK, r) for r in CALIB]
st_cot = layer_stats(model, cot_texts)
st_gen = layer_stats(model, GENERIC)
RESULTS["layer_stats"] = {str(l): {k: v for k, v in s.items() if k != "dim_max"} for l, s in st_cot.items()}
RESULTS["thresholds"] = dict(cot=[st_cot[l]["p995"] for l in range(L)], generic=[st_gen[l]["p995"] for l in range(L)])
save_results()

C3_COT = {l: clamp_hook(st_cot[l]["p995"]) for l in range(L)}
C3_GEN = {l: clamp_hook(st_gen[l]["p995"]) for l in range(L)}
RUNS["c3_cot"] = run_config("c3_cot", model, MAIN, C3_COT)
RUNS["c3_gen"] = run_config("c3_gen", model, MAIN, C3_GEN)

# Ablations on the fixed subset (all use CoT thresholds)
RUNS["nf4_abl"] = [r for r in RUNS["nf4"] if r["idx"] in {a["idx"] for a in ABL}]
abl = {}
for l in CFG["single_layers"]:
    if l < L:
        abl[f"single_{l}"] = run_config(f"abl_single_{l}", model, ABL, {l: C3_COT[l]})
for k in CFG["first_k"]:
    if k <= L:
        abl[f"first_{k}"] = run_config(f"abl_first_{k}", model, ABL, {l: C3_COT[l] for l in range(k)})
exempt_fns = {}
for l in range(L):
    mask = torch.zeros_like(st_cot[l]["dim_max"], dtype=torch.bool)
    mask[torch.topk(st_cot[l]["dim_max"], CFG["exempt_top_m"]).indices] = True
    exempt_fns[l] = clamp_hook(st_cot[l]["p995"], mask)
abl["exempt_top_dims"] = run_config("abl_exempt", model, ABL, exempt_fns)

constrained = {}
for kk in CFG["constrained_k"]:
    fns, thr = {}, []
    for l in range(L):   # joint calibration: layer l calibrated with clamps on layers < l active
        s = layer_stats(model, cot_texts, only=l, clamp_fns=fns)[l]
        C = max(s["p995"], s["mean"] + kk * s["std"])
        thr.append(C)
        fns[l] = clamp_hook(C)
    constrained[kk] = fns
    RESULTS.setdefault("thresholds", {})[f"constrained_k{kk}"] = thr
    abl[f"constrained_k{kk}"] = run_config(f"abl_constrained_k{kk}", model, ABL, fns)

if "kl" not in RESULTS:
    RESULTS["kl"] = dict(c3_cot=mean_kl(model, C3_COT), c3_gen=mean_kl(model, C3_GEN),
                         exempt_top_dims=mean_kl(model, exempt_fns),
                         **{f"constrained_k{kk}": mean_kl(model, fns) for kk, fns in constrained.items()},
                         **{f"single_{l}": mean_kl(model, {l: C3_COT[l]}) for l in CFG["single_layers"] if l < L})
RESULTS["c3_samples"] = [r["text"][:600] for r in RUNS["c3_cot"][:3]]
RUNS.update({f"abl_{k}": v for k, v in abl.items()})
save_results()
free(model)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[nf4] acc=56.2% n=16 {'seconds': 28.1, 'footprint_gb': 1.122}
[c3_cot] acc=0.0% n=16 {'seconds': 29.5, 'footprint_gb': 1.122}
[c3_gen] acc=0.0% n=16 {'seconds': 28.8, 'footprint_gb': 1.122}
[abl_single_0] acc=62.5% n=8 {'seconds': 29.7, 'footprint_gb': 1.122}
[abl_single_1] acc=25.0% n=8 {'seconds': 31.5, 'footprint_gb': 1.122}
[abl_single_2] acc=0.0% n=8 {'seconds': 31.7, 'footprint_gb': 1.122}
[abl_single_4] acc=0.0% n=8 {'seconds': 31.5, 'footprint_gb': 1.122}
[abl_single_14] acc=0.0% n=8 {'seconds': 30.5, 'footprint_gb': 1.122}
[abl_single_27] acc=50.0% n=8 {'seconds': 30.3, 'footprint_gb': 1.122}
[abl_first_2] acc=0.0% n=8 {'seconds': 30.2, 'footprint_gb': 1.122}
[abl_first_4] acc=0.0% n=8 {'seconds': 30.9, 'footprint_gb': 1.122}
[abl_first_8] acc=0.0% n=8 {'seconds': 30.0, 'footprint_gb': 1.122}
[abl_first_16] acc=0.0% n=8 {'seconds': 30.1, 'footprint_gb': 1.122}
[abl_exempt] acc=0.0% n=8 {'seconds': 35.3, 'footprint_gb': 1.122}
[abl_constrained_k3] acc=0.0% n=8 {'seconds': 30.4,

In [ ]:
# Load a fresh NF4 model for calibration
model = load_model("nf4")

# Prepare calibration texts
cot_texts = [trace_text(TOK, r) for r in CALIB]

# Recalculate activation statistics using updated function
st_cot = layer_stats(model, cot_texts)
st_gen = layer_stats(model, GENERIC)

# Extract thresholds
cot_thresholds = [st_cot[l]["p995"] for l in range(len(layers_of(model)))]
gen_thresholds = [st_gen[l]["p995"] for l in range(len(layers_of(model)))]

# Display results
print("Number of layers:", len(cot_thresholds))
print("CoT thresholds:", cot_thresholds)
print("Generic thresholds:", gen_thresholds)

# Check for invalid values
print("\nAll CoT thresholds finite:",
      all(torch.isfinite(torch.tensor(x)) for x in cot_thresholds))

print("All Generic thresholds finite:",
      all(torch.isfinite(torch.tensor(x)) for x in gen_thresholds))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Number of layers: 28
CoT thresholds: [2.345703125, 4.11328125, 4.640625, 5.109375, 5.44140625, 5.703125, 5.96484375, 6.21484375, 6.234375, 6.33203125, 6.48046875, 6.66796875, 6.88671875, 7.12890625, 7.28125, 7.36328125, 7.41015625, 7.69140625, 8.1875, 8.9921875, 9.40625, 10.7421875, 11.96875, 13.1015625, 14.609375, 16.125, 17.421875, 21.84375]
Generic thresholds: [2.255859375, 3.95703125, 4.609375, 5.234375, 5.66015625, 5.85546875, 6.1875, 6.65625, 6.83203125, 7.02734375, 7.13671875, 7.4609375, 7.7578125, 8.15625, 8.1796875, 8.421875, 8.625, 8.953125, 9.5078125, 10.5234375, 11.09375, 12.8828125, 14.5703125, 16.203125, 18.15625, 19.8125, 21.265625, 39.1875]

All CoT thresholds finite: True
All Generic thresholds finite: True


In [ ]:
for l in range(len(cot_thresholds)):
    print(
        f"Layer {l:02d} | "
        f"CoT: {cot_thresholds[l]:.4f} | "
        f"Generic: {gen_thresholds[l]:.4f}"
    )

Layer 00 | CoT: 2.3457 | Generic: 2.2559
Layer 01 | CoT: 4.1133 | Generic: 3.9570
Layer 02 | CoT: 4.6406 | Generic: 4.6094
Layer 03 | CoT: 5.1094 | Generic: 5.2344
Layer 04 | CoT: 5.4414 | Generic: 5.6602
Layer 05 | CoT: 5.7031 | Generic: 5.8555
Layer 06 | CoT: 5.9648 | Generic: 6.1875
Layer 07 | CoT: 6.2148 | Generic: 6.6562
Layer 08 | CoT: 6.2344 | Generic: 6.8320
Layer 09 | CoT: 6.3320 | Generic: 7.0273
Layer 10 | CoT: 6.4805 | Generic: 7.1367
Layer 11 | CoT: 6.6680 | Generic: 7.4609
Layer 12 | CoT: 6.8867 | Generic: 7.7578
Layer 13 | CoT: 7.1289 | Generic: 8.1562
Layer 14 | CoT: 7.2812 | Generic: 8.1797
Layer 15 | CoT: 7.3633 | Generic: 8.4219
Layer 16 | CoT: 7.4102 | Generic: 8.6250
Layer 17 | CoT: 7.6914 | Generic: 8.9531
Layer 18 | CoT: 8.1875 | Generic: 9.5078
Layer 19 | CoT: 8.9922 | Generic: 10.5234
Layer 20 | CoT: 9.4062 | Generic: 11.0938
Layer 21 | CoT: 10.7422 | Generic: 12.8828
Layer 22 | CoT: 11.9688 | Generic: 14.5703
Layer 23 | CoT: 13.1016 | Generic: 16.2031
Layer 24

## Stage 4 — C1 with a module audit, and the combined configuration

## Stage 5 — C2 verifier (offline: NF4 outputs, INT8 regeneration for flagged items)

In [ ]:
def audit_blocks(model):
    """Module class names per decoder block: proves which blocks are really quantized."""
    per_block = {}
    for name, mod in model.named_modules():
        m = re.match(r"model\.layers\.(\d+)\.", name + ".")
        if m and isinstance(mod, torch.nn.Linear):
            per_block.setdefault(int(m.group(1)), set()).add(type(mod).__name__)
    return {b: sorted(v) for b, v in sorted(per_block.items())}


def is_fp_block(types):
    return types == ["Linear"]


protected = RESULTS["sensitivity"]["protected"]
candidates = [
    ["lm_head"] + [rf"model\.layers\.{b}\." for b in protected],                    # anchored regex
    ["lm_head"] + [f"model.layers.{b}." for b in protected],                        # dotted prefix
    ["lm_head"] + [f"model.layers.{b}.{m}" for b in protected for m in
                   ["self_attn.q_proj", "self_attn.k_proj", "self_attn.v_proj", "self_attn.o_proj",
                    "mlp.gate_proj", "mlp.up_proj", "mlp.down_proj"]],                 # explicit names
]
model, audit_ok = None, False
for skip in candidates:
    if model is not None:
        free(model)
    model = load_model("nf4", skip=skip)
    audit = audit_blocks(model)
    fp_blocks = sorted(b for b, t in audit.items() if is_fp_block(t))
    audit_ok = fp_blocks == sorted(protected)
    print("skip list", skip[:3], "... fp16 blocks:", fp_blocks, "OK" if audit_ok else "MISMATCH")
    if audit_ok:
        break
RESULTS["c1_audit"] = dict(protected=protected, fp16_blocks=fp_blocks, ok=audit_ok, skip_list=skip,
                           module_types={str(b): t for b, t in audit.items()},
                           footprint_gb=footprint_gb(model))
save_results()
assert audit_ok or TEST_MODE, "C1 could not be applied: protected blocks are still quantized. Do not report C1."
RUNS["c1"] = run_config("c1", model, MAIN)
comb_fns = {l: C3_COT[l] for l in range(L) if l not in protected}
RUNS["combined"] = run_config("combined", model, MAIN, comb_fns)
free(model)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

skip list ['lm_head', 'model\\.layers\\.2\\.', 'model\\.layers\\.4\\.'] ... fp16 blocks: [2, 4, 5, 6, 7, 12, 13] OK
[c1] acc=68.8% n=16 {'seconds': 26.0, 'footprint_gb': 1.613}
[combined] acc=0.0% n=16 {'seconds': 26.9, 'footprint_gb': 1.613}


In [ ]:
# Compare NF4 and C3 outputs on the same questions

for i in range(min(3, len(RUNS["nf4"]))):
    nf4 = RUNS["nf4"][i]
    cot = RUNS["c3_cot"][i]
    gen = RUNS["c3_gen"][i]

    print(f"\n{'='*70}")
    print(f"QUESTION {i+1}")
    print(f"Gold answer: {nf4['gold']}")

    print("\n--- NF4 ---")
    print("Prediction:", nf4["pred"])
    print("Correct:", nf4["correct"])
    print("Response:", nf4["text"][:700])

    print("\n--- C3 CoT ---")
    print("Prediction:", cot["pred"])
    print("Correct:", cot["correct"])
    print("Response:", cot["text"][:700])

    print("\n--- C3 Generic ---")
    print("Prediction:", gen["pred"])
    print("Correct:", gen["correct"])
    print("Response:", gen["text"][:700])


QUESTION 1
Gold answer: 109.0

--- NF4 ---
Prediction: 109.0
Correct: True
Response: Let's denote Darrell's age as \(7x\) and Allen's age as \(11x\), where \(x\) is a common multiplier. According to the problem, the sum of their ages is 162, so we can write the equation:

\[7x + 11x = 162\]

Combining like terms, we get:

\[18x = 162\]

To find \(x\), we divide both sides by 18:

\[x = \frac{162}{18} = 9\]

Now that we have \(x\), we can find the current ages of both Darrell and Allen:

- Darrell's age is \(7x = 7 \times 9 = 63\)
- Allen's age is \(11x = 11 \times 9 = 99\)

To find Allen's age 10 years from now, we add 10 to his current age:

\[99 + 10 = 109\]

Therefore, Allen's age 10 years from now is \(\boxed{109}\).

--- C3 CoT ---
Prediction: 2.0
Correct: False
Response:  162.12. is 12. is 12.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.22,2.2,22,2,22,2,2,222,2,22,2,22,2,22,2,22,2,2.2,2.2,2.22,2,22,2,2,

In [ ]:
print("Number of decoder layers:", len(st_cot))

print("\nC3 CoT thresholds:")
for layer, stats in st_cot.items():
    print(
        f"Layer {layer:2d} | "
        f"p995 = {stats['p995']:.6f} | "
        f"mean = {stats['mean']:.6f} | "
        f"std = {stats['std']:.6f} | "
        f"max = {stats['max']:.6f}"
    )

print("\nC3 Generic thresholds:")
for layer, stats in st_gen.items():
    print(
        f"Layer {layer:2d} | "
        f"p995 = {stats['p995']:.6f} | "
        f"mean = {stats['mean']:.6f} | "
        f"std = {stats['std']:.6f} | "
        f"max = {stats['max']:.6f}"
    )

Number of decoder layers: 28

C3 CoT thresholds:
Layer  0 | p995 = 2.345703 | mean = 0.319537 | std = 0.385183 | max = 14.781250
Layer  1 | p995 = 4.113281 | mean = 0.494127 | std = 8.818286 | max = 2800.000000
Layer  2 | p995 = 4.640625 | mean = 0.647027 | std = 17.926336 | max = 5664.000000
Layer  3 | p995 = 5.109375 | mean = 0.732076 | std = 18.298862 | max = 5776.000000
Layer  4 | p995 = 5.441406 | mean = 0.792027 | std = 18.526350 | max = 5844.000000
Layer  5 | p995 = 5.703125 | mean = 0.750881 | std = 18.800352 | max = 5932.000000
Layer  6 | p995 = 5.964844 | mean = 0.770168 | std = 18.809315 | max = 5936.000000
Layer  7 | p995 = 6.214844 | mean = 0.782451 | std = 18.816906 | max = 5940.000000
Layer  8 | p995 = 6.234375 | mean = 0.787860 | std = 18.818012 | max = 5940.000000
Layer  9 | p995 = 6.332031 | mean = 0.824959 | std = 18.830240 | max = 5944.000000
Layer 10 | p995 = 6.480469 | mean = 0.828304 | std = 18.834066 | max = 5948.000000
Layer 11 | p995 = 6.667969 | mean = 0.8526

In [ ]:
@torch.no_grad()
def clipping_diagnostic(model, text, thresholds):
    L = len(layers_of(model))
    stats = {}

    def make_hook(l):
        def fn(mod, inp, out):
            h = _hidden(out).detach().float()

            C = thresholds[l]

            clipped = h.abs() > C

            stats[l] = {
                "threshold": float(C),
                "clipped_pct": float(clipped.float().mean().item() * 100),
                "max_before": float(h.abs().max().item()),
                "mean_abs": float(h.abs().mean().item()),
            }

        return fn

    fns = {l: make_hook(l) for l in range(L)}

    with hooks(model, fns):
        ids = TOK(text, return_tensors="pt").input_ids.to(model.device)
        model(input_ids=ids)

    return stats


model = load_model("nf4")

diag_cot = clipping_diagnostic(
    model,
    cot_texts[0],
    {l: st_cot[l]["p995"] for l in range(L)}
)

print("Layer | Threshold | Clipped % | Max activation")
print("-" * 55)

for l, s in diag_cot.items():
    print(
        f"{l:5d} | "
        f"{s['threshold']:9.3f} | "
        f"{s['clipped_pct']:8.3f}% | "
        f"{s['max_before']:14.2f}"
    )

free(model)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Layer | Threshold | Clipped % | Max activation
-------------------------------------------------------
    0 |     2.346 |    0.470% |          14.63
    1 |     4.113 |    0.438% |        2800.00
    2 |     4.641 |    0.438% |        5664.00
    3 |     5.109 |    0.439% |        5776.00
    4 |     5.441 |    0.432% |        5844.00
    5 |     5.703 |    0.439% |        5932.00
    6 |     5.965 |    0.439% |        5936.00
    7 |     6.215 |    0.440% |        5940.00
    8 |     6.234 |    0.438% |        5940.00
    9 |     6.332 |    0.438% |        5944.00
   10 |     6.480 |    0.446% |        5948.00
   11 |     6.668 |    0.451% |        5952.00
   12 |     6.887 |    0.450% |        5952.00
   13 |     7.129 |    0.445% |        5952.00
   14 |     7.281 |    0.442% |        5952.00
   15 |     7.363 |    0.446% |        5952.00
   16 |     7.410 |    0.448% |        5952.00
   17 |     7.691 |    0.451% |        5952.00
   18 |     8.188 |    0.452% |        5956.00
   1

In [ ]:
@torch.no_grad()
def compare_clipping_impact(model, text, clamp_fns):
    ids = TOK(
        text,
        return_tensors="pt"
    ).input_ids.to(model.device)

    # Original NF4 output
    base_logits = model(input_ids=ids).logits.float()

    # Output with clipping
    with hooks(model, clamp_fns):
        clamp_logits = model(input_ids=ids).logits.float()

    # Probability distributions
    p = torch.softmax(base_logits, dim=-1)
    q = torch.softmax(clamp_logits, dim=-1)

    # KL divergence for every token
    kl = (
        p * (
            torch.log_softmax(base_logits, dim=-1)
            - torch.log_softmax(clamp_logits, dim=-1)
        )
    ).sum(dim=-1)

    # Compare next-token predictions
    base_next = base_logits[:, -1, :].argmax(dim=-1)
    clamp_next = clamp_logits[:, -1, :].argmax(dim=-1)

    return {
        "mean_kl": kl.mean().item(),
        "max_kl": kl.max().item(),
        "next_token_changed": (
            base_next != clamp_next
        ).item(),
        "base_next_token": TOK.decode(base_next),
        "clamped_next_token": TOK.decode(clamp_next),
    }


model = load_model("nf4")

print("C3 CoT:")
print(
    compare_clipping_impact(
        model,
        cot_texts[0],
        C3_COT
    )
)

print("\nC3 Generic:")
print(
    compare_clipping_impact(
        model,
        cot_texts[0],
        C3_GEN
    )
)

free(model)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

C3 CoT:
{'mean_kl': 8.049201011657715, 'max_kl': 25.627185821533203, 'next_token_changed': True, 'base_next_token': '<|endoftext|>', 'clamped_next_token': ' '}

C3 Generic:
{'mean_kl': 7.346240997314453, 'max_kl': 24.687118530273438, 'next_token_changed': True, 'base_next_token': '<|endoftext|>', 'clamped_next_token': ' '}


In [ ]:
@torch.no_grad()
def layerwise_kl_diagnostic(model, text, thresholds):
    ids = TOK(
        text,
        return_tensors="pt"
    ).input_ids.to(model.device)

    # Reference output without clipping
    base_logits = model(input_ids=ids).logits.float()
    base_logp = torch.log_softmax(base_logits, dim=-1)
    base_p = base_logp.exp()

    results = []

    for layer, threshold in thresholds.items():

        # Apply clipping to only one layer
        single_hook = {
            layer: clamp_hook(threshold)
        }

        with hooks(model, single_hook):
            logits = model(input_ids=ids).logits.float()

        logq = torch.log_softmax(logits, dim=-1)

        kl = (
            base_p * (base_logp - logq)
        ).sum(dim=-1).mean().item()

        results.append({
            "layer": layer,
            "threshold": threshold,
            "mean_kl": kl
        })

    return sorted(
        results,
        key=lambda x: x["mean_kl"],
        reverse=True
    )


model = load_model("nf4")

layerwise_results = layerwise_kl_diagnostic(
    model,
    cot_texts[0],
    {l: st_cot[l]["p995"] for l in range(L)}
)

print("Layer | Threshold | Mean KL")
print("-" * 40)

for r in layerwise_results:
    print(
        f"{r['layer']:5d} | "
        f"{r['threshold']:9.3f} | "
        f"{r['mean_kl']:8.4f}"
    )

free(model)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Layer | Threshold | Mean KL
----------------------------------------
   13 |     7.129 |   5.3558
   12 |     6.887 |   5.1332
    2 |     4.641 |   4.6932
   14 |     7.281 |   4.4838
    6 |     5.965 |   4.4058
    7 |     6.215 |   4.2710
    5 |     5.703 |   3.9869
    8 |     6.234 |   3.9847
    4 |     5.441 |   3.9717
   15 |     7.363 |   3.8742
    3 |     5.109 |   3.6055
   16 |     7.410 |   3.5837
   11 |     6.668 |   3.5646
    9 |     6.332 |   3.5233
   17 |     7.691 |   3.4370
   10 |     6.480 |   3.0215
   18 |     8.188 |   2.5029
   19 |     8.992 |   1.4002
    1 |     4.113 |   1.0204
   20 |     9.406 |   0.6357
   21 |    10.742 |   0.5211
   24 |    14.609 |   0.4460
   22 |    11.969 |   0.4101
   23 |    13.102 |   0.3139
   25 |    16.125 |      nan
   26 |    17.422 |   1.5164
    0 |     2.346 |   0.2044
   27 |    21.844 |   0.1225


In [ ]:
@torch.no_grad()
def check_layer_stability(model, text, thresholds):
    ids = TOK(
        text,
        return_tensors="pt"
    ).input_ids.to(model.device)

    base_logits = model(input_ids=ids).logits.float()

    results = []

    for layer, threshold in thresholds.items():
        with hooks(model, {layer: clamp_hook(threshold)}):
            logits = model(input_ids=ids).logits.float()

        results.append({
            "layer": layer,
            "finite": bool(torch.isfinite(logits).all().item()),
            "nan_count": int(torch.isnan(logits).sum().item()),
            "inf_count": int(torch.isinf(logits).sum().item()),
            "max_abs_logit": (
                float(logits.abs().max().item())
                if torch.isfinite(logits).all()
                else float("nan")
            ),
            "logits_changed": not torch.allclose(
                base_logits,
                logits,
                rtol=1e-4,
                atol=1e-4
            )
        })

    return results


model = load_model("nf4")

stability = check_layer_stability(
    model,
    cot_texts[0],
    {l: st_cot[l]["p995"] for l in range(L)}
)

print("Layer | Finite | NaNs | Infs | Max abs logit")
print("-" * 60)

for r in stability:
    print(
        f"{r['layer']:5d} | "
        f"{str(r['finite']):6s} | "
        f"{r['nan_count']:4d} | "
        f"{r['inf_count']:4d} | "
        f"{r['max_abs_logit']:14.4f}"
    )

free(model)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Layer | Finite | NaNs | Infs | Max abs logit
------------------------------------------------------------
    0 | True   |    0 |    0 |        41.2188
    1 | True   |    0 |    0 |        34.2812
    2 | True   |    0 |    0 |        25.4375
    3 | True   |    0 |    0 |        26.1094
    4 | True   |    0 |    0 |        24.8906
    5 | True   |    0 |    0 |        23.8906
    6 | True   |    0 |    0 |        25.1094
    7 | True   |    0 |    0 |        24.6719
    8 | True   |    0 |    0 |        26.7031
    9 | True   |    0 |    0 |        28.9219
   10 | True   |    0 |    0 |        28.0312
   11 | True   |    0 |    0 |        26.3438
   12 | True   |    0 |    0 |        28.7188
   13 | True   |    0 |    0 |        34.8438
   14 | True   |    0 |    0 |        32.1875
   15 | True   |    0 |    0 |        34.8125
   16 | True   |    0 |    0 |        44.8125
   17 | True   |    0 |    0 |        37.1250
   18 | True   |    0 |    0 |        34.9375
   19 | True   |    

In [ ]:
@torch.no_grad()
def inspect_layer25(model, text, threshold):
    ids = TOK(
        text,
        return_tensors="pt"
    ).input_ids.to(model.device)

    layer = layers_of(model)[25]
    report = {}

    def inspect_before(mod, inp):
        h = inp[0].detach().float()

        report["input_finite"] = bool(
            torch.isfinite(h).all().item()
        )
        report["input_nan"] = int(
            torch.isnan(h).sum().item()
        )
        report["input_max"] = float(
            h.abs().max().item()
        )

    def inspect_after(mod, inp, out):
        h = _hidden(out).detach().float()

        report["output_finite"] = bool(
            torch.isfinite(h).all().item()
        )
        report["output_nan"] = int(
            torch.isnan(h).sum().item()
        )
        report["output_max"] = (
            float(h.abs().max().item())
            if torch.isfinite(h).all()
            else float("nan")
        )

    # Baseline forward pass
    baseline = model(input_ids=ids).logits.float()

    report["baseline_logits_finite"] = bool(
        torch.isfinite(baseline).all().item()
    )

    # Inspect Layer 25 without clamping
    pre_hook = layer.register_forward_pre_hook(inspect_before)
    post_hook = layer.register_forward_hook(inspect_after)

    try:
        model(input_ids=ids)
    finally:
        pre_hook.remove()
        post_hook.remove()

    # Inspect Layer 25 with clamping
    def clamp_and_inspect(mod, inp, out):
        h = _hidden(out).detach().float()

        report["clamp_input_finite"] = bool(
            torch.isfinite(h).all().item()
        )
        report["clamp_input_nan"] = int(
            torch.isnan(h).sum().item()
        )

        hc = h.clamp(-threshold, threshold)

        report["clamp_output_finite"] = bool(
            torch.isfinite(hc).all().item()
        )
        report["clamp_output_nan"] = int(
            torch.isnan(hc).sum().item()
        )

        return _out(out, hc.to(_hidden(out).dtype))

    with hooks(model, {25: clamp_and_inspect}):
        clamped_logits = model(input_ids=ids).logits.float()

    report["clamped_logits_finite"] = bool(
        torch.isfinite(clamped_logits).all().item()
    )
    report["clamped_logits_nan"] = int(
        torch.isnan(clamped_logits).sum().item()
    )

    return report


model = load_model("nf4")

report = inspect_layer25(
    model,
    cot_texts[0],
    st_cot[25]["p995"]
)

for key, value in report.items():
    print(f"{key}: {value}")

free(model)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

baseline_logits_finite: True
input_finite: True
input_nan: 0
input_max: 5928.0
output_finite: True
output_nan: 0
output_max: 5928.0
clamp_input_finite: True
clamp_input_nan: 0
clamp_output_finite: True
clamp_output_nan: 0
clamped_logits_finite: False
clamped_logits_nan: 33729792


In [ ]:
@torch.no_grad()
def trace_nan_layers(model, text, target_layer=25, threshold=None):
    ids = TOK(text, return_tensors="pt").input_ids.to(model.device)

    layers = layers_of(model)
    report = {}

    def make_hook(i):
        def hook(mod, inp, out):
            h = _hidden(out).detach().float()

            report[i] = {
                "finite": bool(torch.isfinite(h).all().item()),
                "nan": int(torch.isnan(h).sum().item()),
                "inf": int(torch.isinf(h).sum().item()),
                "max_abs": (
                    float(h.abs().max().item())
                    if torch.isfinite(h).all()
                    else float("nan")
                )
            }

            if i == target_layer:
                h = h.clamp(-threshold, threshold)
                return _out(out, h.to(_hidden(out).dtype))

        return hook

    fns = {
        i: make_hook(i)
        for i in range(len(layers))
    }

    with hooks(model, fns):
        logits = model(input_ids=ids).logits.float()

    print("Final logits finite:", bool(torch.isfinite(logits).all().item()))

    print("\nLayer-wise trace:")
    for i, stats in report.items():
        print(i, stats)

    failed = [
        i for i, stats in report.items()
        if not stats["finite"]
    ]

    print("\nFirst failing layer:", failed[0] if failed else "None")

    return report


model = load_model("nf4")

trace = trace_nan_layers(
    model,
    cot_texts[0],
    target_layer=25,
    threshold=st_cot[25]["p995"]
)

free(model)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Final logits finite: False

Layer-wise trace:
0 {'finite': True, 'nan': 0, 'inf': 0, 'max_abs': 14.6328125}
1 {'finite': True, 'nan': 0, 'inf': 0, 'max_abs': 2800.0}
2 {'finite': True, 'nan': 0, 'inf': 0, 'max_abs': 5664.0}
3 {'finite': True, 'nan': 0, 'inf': 0, 'max_abs': 5776.0}
4 {'finite': True, 'nan': 0, 'inf': 0, 'max_abs': 5844.0}
5 {'finite': True, 'nan': 0, 'inf': 0, 'max_abs': 5932.0}
6 {'finite': True, 'nan': 0, 'inf': 0, 'max_abs': 5936.0}
7 {'finite': True, 'nan': 0, 'inf': 0, 'max_abs': 5940.0}
8 {'finite': True, 'nan': 0, 'inf': 0, 'max_abs': 5940.0}
9 {'finite': True, 'nan': 0, 'inf': 0, 'max_abs': 5944.0}
10 {'finite': True, 'nan': 0, 'inf': 0, 'max_abs': 5948.0}
11 {'finite': True, 'nan': 0, 'inf': 0, 'max_abs': 5952.0}
12 {'finite': True, 'nan': 0, 'inf': 0, 'max_abs': 5952.0}
13 {'finite': True, 'nan': 0, 'inf': 0, 'max_abs': 5952.0}
14 {'finite': True, 'nan': 0, 'inf': 0, 'max_abs': 5952.0}
15 {'finite': True, 'nan': 0, 'inf': 0, 'max_abs': 5952.0}
16 {'finite': Tr

In [ ]:
@torch.no_grad()
def inspect_block26(model, text, threshold):
    ids = TOK(text, return_tensors="pt").input_ids.to(model.device)
    block = layers_of(model)[26]

    report = {}

    def make_hook(name):
        def hook(mod, inp, out):
            h = _hidden(out).detach().float()

            report[name] = {
                "finite": bool(torch.isfinite(h).all().item()),
                "nan": int(torch.isnan(h).sum().item()),
                "inf": int(torch.isinf(h).sum().item()),
                "max_abs": (
                    float(h.abs().max().item())
                    if torch.isfinite(h).all()
                    else float("nan")
                )
            }
        return hook

    handles = []

    for name, module in block.named_modules():
        if name == "":
            continue
        handles.append(
            module.register_forward_hook(make_hook(name))
        )

    def clamp25(mod, inp, out):
        h = _hidden(out)
        return _out(out, h.clamp(-threshold, threshold))

    try:
        with hooks(model, {25: clamp25}):
            logits = model(input_ids=ids).logits.float()
    finally:
        for handle in handles:
            handle.remove()

    print("Final logits finite:", bool(torch.isfinite(logits).all().item()))
    print("\nLayer 26 submodule trace:")

    for name, stats in report.items():
        print(name, stats)

    failed = [
        name for name, stats in report.items()
        if not stats["finite"]
    ]

    print("\nFirst failing submodule:", failed[0] if failed else "None")

    return report


model = load_model("nf4")

block26_report = inspect_block26(
    model,
    cot_texts[0],
    st_cot[25]["p995"]
)

free(model)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Final logits finite: False

Layer 26 submodule trace:
input_layernorm {'finite': True, 'nan': 0, 'inf': 0, 'max_abs': 19.515625}
self_attn.q_proj {'finite': True, 'nan': 0, 'inf': 0, 'max_abs': 15.4296875}
self_attn.k_proj {'finite': True, 'nan': 0, 'inf': 0, 'max_abs': 15.5703125}
self_attn.v_proj {'finite': True, 'nan': 0, 'inf': 0, 'max_abs': 24.625}
self_attn.o_proj {'finite': True, 'nan': 0, 'inf': 0, 'max_abs': 41.46875}
self_attn {'finite': True, 'nan': 0, 'inf': 0, 'max_abs': 41.46875}
post_attention_layernorm {'finite': True, 'nan': 0, 'inf': 0, 'max_abs': 10.2890625}
mlp.gate_proj {'finite': True, 'nan': 0, 'inf': 0, 'max_abs': 241.375}
mlp.act_fn {'finite': True, 'nan': 0, 'inf': 0, 'max_abs': 241.375}
mlp.up_proj {'finite': True, 'nan': 0, 'inf': 0, 'max_abs': 299.25}
mlp.down_proj {'finite': False, 'nan': 811, 'inf': 725, 'max_abs': nan}
mlp {'finite': False, 'nan': 811, 'inf': 725, 'max_abs': nan}

First failing submodule: mlp.down_proj


In [ ]:
@torch.no_grad()
def inspect_down_proj(model, text, threshold):
    ids = TOK(text, return_tensors="pt").input_ids.to(model.device)

    block = layers_of(model)[26]
    down = block.mlp.down_proj

    report = {}

    def inspect_input(mod, inp):
        x = inp[0].detach().float()

        report["input"] = {
            "finite": bool(torch.isfinite(x).all().item()),
            "nan": int(torch.isnan(x).sum().item()),
            "inf": int(torch.isinf(x).sum().item()),
            "max_abs": float(x.abs().max().item()),
            "mean_abs": float(x.abs().mean().item()),
            "shape": tuple(x.shape),
            "dtype": str(inp[0].dtype)
        }

    def inspect_output(mod, inp, out):
        y = _hidden(out).detach().float()

        report["output"] = {
            "finite": bool(torch.isfinite(y).all().item()),
            "nan": int(torch.isnan(y).sum().item()),
            "inf": int(torch.isinf(y).sum().item()),
            "max_abs": (
                float(y.abs().max().item())
                if torch.isfinite(y).all()
                else float("nan")
            ),
            "dtype": str(_hidden(out).dtype)
        }

    report["weight_dtype"] = str(down.weight.dtype)
    report["weight_shape"] = tuple(down.weight.shape)
    report["weight_class"] = type(down).__name__

    if hasattr(down, "compute_dtype"):
        report["compute_dtype"] = str(down.compute_dtype)

    h1 = down.register_forward_pre_hook(inspect_input)
    h2 = down.register_forward_hook(inspect_output)

    def clamp25(mod, inp, out):
        h = _hidden(out)
        return _out(out, h.clamp(-threshold, threshold))

    try:
        with hooks(model, {25: clamp25}):
            model(input_ids=ids)
    finally:
        h1.remove()
        h2.remove()

    for key, value in report.items():
        print(key, value)

    return report


model = load_model("nf4")

down_report = inspect_down_proj(
    model,
    cot_texts[0],
    st_cot[25]["p995"]
)

free(model)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

weight_dtype torch.uint8
weight_shape (6881280, 1)
weight_class Linear4bit
compute_dtype torch.float16
input {'finite': False, 'nan': 0, 'inf': 1, 'max_abs': inf, 'mean_abs': inf, 'shape': (1, 222, 8960), 'dtype': 'torch.float16'}
output {'finite': False, 'nan': 811, 'inf': 725, 'max_abs': nan, 'dtype': 'torch.float16'}


In [ ]:
@torch.no_grad()
def inspect_mlp_product(model, text, threshold):
    ids = TOK(text, return_tensors="pt").input_ids.to(model.device)
    mlp = layers_of(model)[26].mlp

    captured = {}

    def save_output(name):
        def hook(mod, inp, out):
            captured[name] = out.detach()
        return hook

    h1 = mlp.gate_proj.register_forward_hook(save_output("gate"))
    h2 = mlp.up_proj.register_forward_hook(save_output("up"))

    def clamp25(mod, inp, out):
        h = _hidden(out)
        return _out(out, h.clamp(-threshold, threshold))

    try:
        with hooks(model, {25: clamp25}):
            model(input_ids=ids)
    finally:
        h1.remove()
        h2.remove()

    gate = captured["gate"]
    up = captured["up"]

    # Reconstruct the MLP intermediate
    activated = mlp.act_fn(gate)

    product_fp16 = activated * up
    product_fp32 = activated.float() * up.float()

    for name, tensor in [
        ("gate", gate),
        ("up", up),
        ("activated_gate", activated),
        ("product_fp16", product_fp16),
        ("product_fp32", product_fp32),
    ]:
        x = tensor.float()

        print(
            name,
            {
                "finite": bool(torch.isfinite(x).all().item()),
                "nan": int(torch.isnan(x).sum().item()),
                "inf": int(torch.isinf(x).sum().item()),
                "max_abs": (
                    float(x.abs().max().item())
                    if torch.isfinite(x).all()
                    else float("nan")
                ),
                "dtype": str(tensor.dtype),
            }
        )

    return captured


model = load_model("nf4")

mlp_debug = inspect_mlp_product(
    model,
    cot_texts[0],
    st_cot[25]["p995"]
)

free(model)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

gate {'finite': True, 'nan': 0, 'inf': 0, 'max_abs': 241.375, 'dtype': 'torch.float16'}
up {'finite': True, 'nan': 0, 'inf': 0, 'max_abs': 299.25, 'dtype': 'torch.float16'}
activated_gate {'finite': True, 'nan': 0, 'inf': 0, 'max_abs': 241.375, 'dtype': 'torch.float16'}
product_fp16 {'finite': False, 'nan': 0, 'inf': 1, 'max_abs': nan, 'dtype': 'torch.float16'}
product_fp32 {'finite': True, 'nan': 0, 'inf': 0, 'max_abs': 72231.46875, 'dtype': 'torch.float32'}


In [ ]:
@torch.no_grad()
def sweep_layer25_threshold(model, text, base_threshold):
    ids = TOK(text, return_tensors="pt").input_ids.to(model.device)

    multipliers = [1, 2, 4, 8, 16, 32, None]

    results = []

    for mult in multipliers:
        if mult is None:
            label = "no_clipping"
            threshold = None
        else:
            label = f"{mult}x"
            threshold = base_threshold * mult

        def intervention(mod, inp, out):
            h = _hidden(out)

            if threshold is None:
                return out

            return _out(out, h.clamp(-threshold, threshold))

        with hooks(model, {25: intervention}):
            output = model(input_ids=ids)

        logits = output.logits.float()

        result = {
            "setting": label,
            "threshold": threshold,
            "logits_finite": bool(torch.isfinite(logits).all().item()),
            "nan": int(torch.isnan(logits).sum().item()),
            "inf": int(torch.isinf(logits).sum().item()),
        }

        results.append(result)
        print(result)

        del output, logits

    return results


model = load_model("nf4")

threshold_sweep = sweep_layer25_threshold(
    model,
    cot_texts[0],
    st_cot[25]["p995"]
)

free(model)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

{'setting': '1x', 'threshold': 16.125, 'logits_finite': False, 'nan': 33729792, 'inf': 0}
{'setting': '2x', 'threshold': 32.25, 'logits_finite': False, 'nan': 33729792, 'inf': 0}
{'setting': '4x', 'threshold': 64.5, 'logits_finite': False, 'nan': 33729792, 'inf': 0}
{'setting': '8x', 'threshold': 129.0, 'logits_finite': True, 'nan': 0, 'inf': 0}
{'setting': '16x', 'threshold': 258.0, 'logits_finite': True, 'nan': 0, 'inf': 0}
{'setting': '32x', 'threshold': 516.0, 'logits_finite': True, 'nan': 0, 'inf': 0}
{'setting': 'no_clipping', 'threshold': None, 'logits_finite': True, 'nan': 0, 'inf': 0}


In [ ]:
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
        ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}


def safe_eval(expr):
    def ev(n):
        if isinstance(n, ast.Expression):
            return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)):
            return float(n.value)
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS:
            if isinstance(n.op, ast.Pow) and abs(ev(n.right)) > 10:
                raise ValueError("exponent too large")
            return _OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in _OPS:
            return _OPS[type(n.op)](ev(n.operand))
        raise ValueError("unsupported")
    return ev(ast.parse(expr, mode="eval"))


def normalize_math(s):
    s = re.sub(r"\\text\{[^{}]*\}", " ", s)
    for _ in range(3):
        s = re.sub(r"\\d?frac\{([^{}]*)\}\{([^{}]*)\}", r"((\1)/(\2))", s)
    for a, b in [("\\times", "*"), ("×", "*"), ("\\cdot", "*"), ("÷", "/"), ("\\div", "/"), ("\\left", ""),
                 ("\\right", ""), ("\\(", " "), ("\\)", " "), ("\\[", " "), ("\\]", " "), ("$", " "), ("\\$", " "),
                 ("−", "-"), ("^", "**"), ("{", "("), ("}", ")")]:
        s = s.replace(a, b)
    s = re.sub(r"(?<=\d),(?=\d{3}\b)", "", s)
    s = re.sub(r"(?<=\d)\s*[xX]\s*(?=\d)", "*", s)
    return s


TAIL = re.compile(r"[\d\.\s\+\-\*/\(\)]+$")
HEAD = re.compile(r"^\s*[\(\s]*-?\d+(?:\.\d+)?(?:\s*(?:\*\*|[\+\-\*/])\s*\(?\s*-?\d+(?:\.\d+)?\s*\)?)*")


def check_statements(text):
    checked, bad = 0, 0
    for line in text.splitlines():
        parts = normalize_math(line).split("=")
        for lhs, rhs in zip(parts, parts[1:]):
            t, h = TAIL.search(lhs), HEAD.match(rhs)
            if not t or not h:
                continue
            le, re_ = t.group().strip(), h.group().strip()
            if not re.search(r"[\d\)]\s*(\*\*|[\+\-\*/])\s*[\(\d]", le):
                continue
            try:
                lv, rv = safe_eval(le), safe_eval(re_)
            except Exception:
                continue
            checked += 1
            dec = len(re_.split(".")[-1]) if "." in re_ and re.fullmatch(r"-?\d+\.\d+", re_) else 0
            tol = max(1e-6 * max(1.0, abs(rv)), 0.5 * 10 ** (-dec) if dec else 0)
            if abs(lv - rv) > tol:
                bad += 1
    return checked, bad


def c2_compose(base_runs, fallback_runs):
    fb = {r["idx"]: r for r in fallback_runs}
    out, n_flag, n_checked = [], 0, 0
    for r in base_runs:
        checked, bad = check_statements(r["text"])
        n_checked += checked
        if bad:
            n_flag += 1
            out.append(dict(fb[r["idx"]], escalated=True, base_correct=r["correct"]))
        else:
            out.append(dict(r, escalated=False, base_correct=r["correct"]))
    rescued = sum(r["escalated"] and r["correct"] and not r["base_correct"] for r in out)
    broken = sum(r["escalated"] and not r["correct"] and r["base_correct"] for r in out)
    return out, dict(escalated=n_flag, escalation_rate=100 * n_flag / len(out), rescued=rescued, broken=broken,
                     statements_checked=n_checked)


RUNS["c2"], RESULTS["c2"] = c2_compose(RUNS["nf4"], RUNS["int8"])
RUNS["combined_c2"], RESULTS["combined_c2"] = c2_compose(RUNS["combined"], RUNS["int8"])
save_results()
print("C2:", RESULTS["c2"])

In [ ]:
print("C3 CoT clamp count:", len(C3_COT))
print("C3 Generic clamp count:", len(C3_GEN))

print("\nFirst 5 C3 CoT thresholds:")
for layer in list(C3_COT.keys())[:5]:
    print(layer, st_cot[layer]["p995"])

print("\nFirst 5 C3 Generic thresholds:")
for layer in list(C3_GEN.keys())[:5]:
    print(layer, st_gen[layer]["p995"])

C3 CoT clamp count: 28
C3 Generic clamp count: 28

First 5 C3 CoT thresholds:
0 2.345703125
1 4.11328125
2 4.640625
3 5.109375
4 5.44140625

First 5 C3 Generic thresholds:
0 2.255859375
1 3.95703125
2 4.609375
3 5.234375
4 5.66015625


## Stage 6 — summary tables and power

In [ ]:
import pandas as pd

In [ ]:
print("Available configurations:")
print(list(RUNS.keys()))

print("\nMissing configurations:")
expected = [
    "fp16", "nf4", "int8", "c1", "c2",
    "c3_cot", "c3_gen", "combined", "combined_c2"
]

missing = [name for name in expected if name not in RUNS]
print(missing)

Available configurations:
['fp16', 'int8', 'nf4', 'c3_cot', 'c3_gen', 'nf4_abl', 'abl_single_0', 'abl_single_1', 'abl_single_2', 'abl_single_4', 'abl_single_14', 'abl_single_27', 'abl_first_2', 'abl_first_4', 'abl_first_8', 'abl_first_16', 'abl_exempt_top_dims', 'abl_constrained_k3', 'abl_constrained_k6', 'c1', 'combined']

Missing configurations:
['c2', 'combined_c2']


In [ ]:
expected = [
    "fp16", "nf4", "int8", "c1", "c2",
    "c3_cot", "c3_gen", "combined", "combined_c2"
]

table = {}

for name in expected:
    if name not in RUNS:
        print(f"Skipping {name}: results not available.")
        continue

    s = summarize(RUNS[name])

    s["vs_nf4"] = (
        mcnemar(RUNS["nf4"], RUNS[name])
        if name != "nf4" and "nf4" in RUNS else None
    )

    s["vs_fp16"] = (
        mcnemar(RUNS["fp16"], RUNS[name])
        if name != "fp16" and "fp16" in RUNS else None
    )

    table[name] = s

display(pd.DataFrame(table).T)

Skipping c2: results not available.
Skipping combined_c2: results not available.


,n,correct,acc,ci,cot_tokens,boxed_rate,hit_max_tokens,vs_nf4,vs_fp16
fp16,16,11,68.75,"[44.40435583474066, 85.83535614521796]",333.0625,93.75,6.25,"{'ref_only': 1, 'other_only': 3, 'p': 0.625, '...",None
nf4,16,9,56.25,"[33.1785563988119, 76.90134759450766]",309.6875,56.25,18.75,None,"{'ref_only': 3, 'other_only': 1, 'p': 0.625, '..."
int8,16,14,87.5,"[63.97717273424131, 96.50225122567595]",319.0625,93.75,0.0,"{'ref_only': 0, 'other_only': 5, 'p': 0.0625, ...","{'ref_only': 0, 'other_only': 3, 'p': 0.25, 'n..."
c1,16,11,68.75,"[44.40435583474066, 85.83535614521796]",316.25,100.0,0.0,"{'ref_only': 0, 'other_only': 2, 'p': 0.5, 'n'...","{'ref_only': 3, 'other_only': 3, 'p': 1.0, 'n'..."
c3_cot,16,0,0.0,"[0.0, 19.36076805344365]",512.0,0.0,100.0,"{'ref_only': 9, 'other_only': 0, 'p': 0.003906...","{'ref_only': 11, 'other_only': 0, 'p': 0.00097..."
c3_gen,16,0,0.0,"[0.0, 19.36076805344365]",512.0,0.0,100.0,"{'ref_only': 9, 'other_only': 0, 'p': 0.003906...","{'ref_only': 11, 'other_only': 0, 'p': 0.00097..."
combined,16,0,0.0,"[0.0, 19.36076805344365]",126.0625,0.0,18.75,"{'ref_only': 9, 'other_only': 0, 'p': 0.003906...","{'ref_only': 11, 'other_only': 0, 'p': 0.00097..."


In [ ]:
for name in ["combined", "c3_cot", "c3_gen"]:
    print("\n" + "=" * 70)
    print("CONFIGURATION:", name)
    print("=" * 70)

    runs = RUNS[name]

    print("Number of records:", len(runs))

    for i, item in enumerate(runs[:2]):
        print(f"\n--- Sample {i + 1} ---")

        if isinstance(item, dict):
            for key, value in item.items():
                print(f"{key}: {str(value)[:1000]}")
        else:
            print(str(item)[:1500])


CONFIGURATION: combined
Number of records: 16

--- Sample 1 ---
idx: 30
gold: 109.0
pred: inf
boxed: False
n_tokens: 512
text: /11 11 11 111111111111111111111111
,; 111 111 111 1111 ynet; \n111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111
correct: False

--- Sample 2 ---
idx: 754
gold: 89.0
pred: 2.024030791230791e+18
boxed: False
n_tokens: 24
text: .LA 2024030791230791000/section
correct: False

CONFIGURATION: c3_cot
Number of records: 16

--- Sample 1 ---
idx: 30
gold: 109.0
pred: 2.0
boxed: False
n_tokens: 512
text:  162.12. is 12. is 12.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2.2

In [ ]:
import torch
import types

def stable_mlp_forward(self, x):
    # Compute gate and up projections
    gate = self.gate_proj(x)
    up = self.up_proj(x)

    # Perform multiplication in FP32 to prevent FP16 overflow
    hidden = torch.nn.functional.silu(gate.float()) * up.float()

    # Convert back to the dtype expected by down_proj
    hidden = hidden.to(self.down_proj.weight.dtype)

    # Final projection
    return self.down_proj(hidden)


def apply_stable_mlp(model):
    count = 0

    for layer in model.model.layers:
        layer.mlp.forward = types.MethodType(
            stable_mlp_forward,
            layer.mlp
        )
        count += 1

    print(f"Stable FP32 MLP multiplication applied to {count} layers.")

    return model

In [ ]:
import torch

print("Available model variables:")

for name, obj in globals().copy().items():
    if isinstance(obj, torch.nn.Module):
        print(f"{name} -> {type(obj).__name__}")

Available model variables:
model -> Qwen2ForCausalLM


In [ ]:
model_c3_fixed = apply_stable_mlp(model)

model_c3_fixed.eval()

print("Stable MLP patch applied successfully.")

FP32 MLP computation applied to 28 layers.
Stable MLP patch applied successfully.


In [ ]:
for i in [0, 13, 25, 26, 27]:
    mlp = model_c3_fixed.model.layers[i].mlp
    print(f"Layer {i}: {mlp.forward.__func__.__name__}")

Layer 0: stable_mlp_forward
Layer 13: stable_mlp_forward
Layer 25: stable_mlp_forward
Layer 26: stable_mlp_forward
Layer 27: stable_mlp_forward


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct"
)

print("Tokenizer loaded successfully!")

Tokenizer loaded successfully!


In [ ]:
import torch

# A simple test question
question = "Janet has 3 apples. She buys 5 more apples. How many apples does she have now?"

# Format the question using Qwen's chat template
messages = [
    {
        "role": "user",
        "content": question
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# Convert prompt into model input
inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model_c3_fixed.device)

print("Input prepared successfully.")

Input prepared successfully.


In [ ]:
import torch

print("Model type:", type(model_c3_fixed).__name__)
print("Model device:", model_c3_fixed.device)
print("Model dtype:", next(model_c3_fixed.parameters()).dtype)

print("\nChecking model parameters...")

for name, param in model_c3_fixed.named_parameters():
    if not torch.isfinite(param).all():
        print("Non-finite parameter detected:", name)
        break
else:
    print("All model parameters are finite.")

Model type: Qwen2ForCausalLM
Model device: cuda:0
Model dtype: torch.float16

Checking model parameters...
All model parameters are finite.


In [ ]:
import torch
import torch.nn.functional as F
import types

def stable_mlp_forward(self, x):
    # Original projections
    gate = self.gate_proj(x)
    up = self.up_proj(x)

    # Keep the intermediate computation in FP32
    hidden = F.silu(gate.float()) * up.float()

    # Perform down_proj in FP32 as well
    output = F.linear(
        hidden,
        self.down_proj.weight.float(),
        self.down_proj.bias.float()
        if self.down_proj.bias is not None else None
    )

    # Convert only the final output back to the input dtype
    return output.to(x.dtype)


def apply_stable_mlp(model):
    count = 0

    for layer in model.model.layers:
        layer.mlp.forward = types.MethodType(
            stable_mlp_forward,
            layer.mlp
        )
        count += 1

    print(f"FP32 MLP computation applied to {count} layers.")
    return model

In [ ]:
model_c3_fixed = apply_stable_mlp(model)
model_c3_fixed.eval()

FP32 MLP computation applied to 28 layers.


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear4bit(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1

In [ ]:
import torch
from transformers import AutoModelForCausalLM

model_clean = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)

model_clean.eval()

print("Clean FP16 model loaded.")
print("Model dtype:", next(model_clean.parameters()).dtype)
print("Model device:", next(model_clean.parameters()).device)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Clean FP16 model loaded.
Model dtype: torch.float16
Model device: cuda:0


In [ ]:
with torch.inference_mode():
    output = model_clean.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

generated_text = tokenizer.decode(
    output[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("Generated Answer:")
print(generated_text)

Generated Answer:
Janet originally had 3 apples and then bought 5 more. To find out how many apples she has in total, you add the two amounts together:

3 (original apples) + 5 (bought apples) = 8 apples

So Janet now has 8 apples.
